In [ ]:
!pip install torch torchaudio transformers
!pip install git+https://github.com/huggingface/parler-tts.git
!pip install soundfile librosa

  Cloning https://github.com/huggingface/parler-tts.git to /tmp/pip-req-build-c675qwal
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/parler-tts.git /tmp/pip-req-build-c675qwal
  Resolved https://github.com/huggingface/parler-tts.git to commit d108732cd57788ec86bc857d99a6cabd66663d68
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/descriptinc/audiotools to /tmp/pip-install-52w3bjfj/descript-audiotools_6b8cdaa8fe14488e82171e20ca4aface
  Running command git clone --filter=blob:none --quiet https://github.com/descriptinc/audiotools /tmp/pip-install-52w3bjfj/descript-audiotools_6b8cdaa8fe14488e82171e20ca4aface
  Resolved https://github.com/descriptinc/audiotools to commit 348ebf2034ce24e2a91a553e3171cb00c0c71678
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.5 MB/s eta 0:00:00
  Prep

In [ ]:
import torch
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer
import soundfile as sf

device = "cuda:0" if torch.cuda.is_available() else "cpu"

# Load Model
model_tts = ParlerTTSForConditionalGeneration.from_pretrained("ai4bharat/indic-parler-tts").to(device)
tokenizer_tts = AutoTokenizer.from_pretrained("ai4bharat/indic-parler-tts")

def text_to_speech(text, description="A female speaker with a clear voice and natural pace."):
    input_ids = tokenizer_tts(description, return_tensors="pt").input_ids.to(device)
    prompt_input_ids = tokenizer_tts(text, return_tensors="pt").input_ids.to(device)

    generation = model_tts.generate(input_ids=input_ids, prompt_input_ids=prompt_input_ids)
    audio_arr = generation.cpu().numpy().squeeze()

    # Save as temporary file for the ASR to read
    sf.write("temp_output.wav", audio_arr, model_tts.config.sampling_rate)
    return "temp_output.wav"

In [ ]:
from transformers import AutoModel, AutoProcessor
import torchaudio

# Load Model
# Note: Ensure you have accepted the terms on Hugging Face if using the 600M model
model_asr = AutoModel.from_pretrained("ai4bharat/indic-conformer-600m-multilingual", trust_remote_code=True).to(device)
processor_asr = AutoProcessor.from_pretrained("ai4bharat/indic-conformer-600m-multilingual")

def speech_to_text(audio_path, language_code="hi"):
    # Load and resample audio to 16kHz (Standard for Conformer)
    speech_array, sampling_rate = torchaudio.load(audio_path)
    if sampling_rate != 16000:
        resampler = torchaudio.transforms.Resample(sampling_rate, 16000)
        speech_array = resampler(speech_array)

    # Prepare inputs
    inputs = processor_asr(speech_array.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Generate transcription
    with torch.no_grad():
        logits = model_asr(**inputs).logits

    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor_asr.batch_decode(predicted_ids)[0]
    return transcription

In [ ]:
def run_pipeline(input_text, lang="hi"):
    print(f"--- Original Text: {input_text} ---")

    # 1. Text -> Audio
    print("Generating audio...")
    audio_file = text_to_speech(input_text)

    # 2. Audio -> Text
    print("Transcribing audio back to text...")
    result_text = speech_to_text(audio_file, language_code=lang)

    print(f"--- Final Result: {result_text} ---")

# Example usage (Hindi)
run_pipeline("नमस्ते, आप कैसे हैं?")